# Durable Workflows with LangGraph

> **The story.** Graph workflow engines make control flow explicit: nodes own work, edges own routing, reducers own concurrent state, and checkpoints own resume semantics. LangGraph applies those ideas to model-driven workflows.
>
> **Where you are.** OrderFlow has typed tools, bounded control, and durable state. PO `#7307` needs parallel inventory and quote checks plus Finance approval. A free-form loop cannot show where to interrupt or resume.
>
> **Notation.** $G=(V,E)$ is the workflow graph; $s$ is typed graph state; $V$ contains nodes; $E$ contains normal or conditional edges; $J$ is the durable event journal.

## 0 - The Challenge

![A purchase order fans into inventory and quote branches, crashes after quote collection, and resumes from a durable state journal at reduction and approval without repeating completed work](../images/ch03-durable-graph-resume.png)

> **The mission:** every high-value PO must visit Finance, low-value POs must not, and a crash after quote collection must resume without repeating that side effect.

```mermaid
flowchart LR
    A["PO-7307"] --> G["Parallel checks"]
    G --> R{ "Approval route" }
    R -->|"High value"| F["Finance interrupt"]
    R -->|"Low value"| L["Automatic path"]
    F --> C["Commit"]
    L --> C
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style R fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style L fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

In [ ]:
# -- Setup: import the deterministic OrderFlow runtime -----------------------
from pathlib import Path
import json
import sys

TRACK_DIR = Path(__vsc_ipynb_file__).resolve().parents[1] if '__vsc_ipynb_file__' in globals() else Path.cwd().resolve().parent
if str(TRACK_DIR) not in sys.path:
    sys.path.insert(0, str(TRACK_DIR))
from concurrent.futures import ThreadPoolExecutor
from copy import deepcopy
from typing import TypedDict

from shared import INVENTORY, SUPPLIER_QUOTES, approval_route, request_by_id

high_value = request_by_id("PO-7307")
low_value = request_by_id("PO-7293")
print("Walking incident:", high_value["email"])


## 1 - Build the State Machine Before the Framework

The mechanism is a transition function over typed state. If the next node cannot be derived from state, the workflow is still hidden in prose.

```mermaid
stateDiagram-v2
    [*] --> gather
    gather --> route
    route --> auto: total <= 5000
    route --> manager: 5000 < total <= 25000
    route --> finance: total > 25000
    auto --> done
    manager --> done
    finance --> done
```


In [ ]:
# -- Implement a framework-free transition table -------------------------
def gather_checks(state):
    request = state["request"]
    inventory = {"available": INVENTORY[request["sku"]]["available"]}
    fresh = [quote for quote in SUPPLIER_QUOTES[request["sku"]] if quote["trusted"] and quote["age_hours"] <= 48]
    quote = min(fresh, key=lambda item: item["unit_price"])
    return {**state, "inventory": inventory, "quote": quote, "gathered": True, "visits": state["visits"] + ["gather"]}


def route_approval(state):
    total = state["quote"]["unit_price"] * state["request"]["quantity"]
    return {**state, "total": total, "route": approval_route(total), "visits": state["visits"] + ["route"]}


def approve(state):
    return {**state, "approved": True, "terminal": "completed", "visits": state["visits"] + [state["route"]]}


def run_state_machine(request):
    state = {"request": deepcopy(request), "visits": [], "gathered": False, "approved": False, "terminal": None}
    state = gather_checks(state)
    state = route_approval(state)
    state = approve(state)
    return state

manual_high = run_state_machine(high_value)
manual_low = run_state_machine(low_value)
print("High-value visits:", manual_high["visits"])
print("Low-value visits:", manual_low["visits"])
assert "finance" in manual_high["visits"] and "finance" not in manual_low["visits"]


## 2 - Fan-Out and Fan-In

Inventory and supplier checks are independent reads. Running them sequentially adds their latency; fan-out makes elapsed latency approach the slower branch, then fan-in merges both observations.

$$
L_{parallel} \approx \max(L_{inventory}, L_{quote}) \le L_{inventory}+L_{quote}
$$

Parallel latency is bounded by the slower branch instead of the sum, ignoring small scheduling overhead.

```mermaid
flowchart LR
    S["Typed state"] --> I["Inventory branch"]
    S --> Q["Quote branch"]
    I --> J["Reducer / fan-in"]
    Q --> J
    J --> R["Approval route"]
    style S fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style I fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style Q fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style J fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style R fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

In [ ]:
# -- Run independent tools concurrently and measure modeled latency -------
def inventory_branch(request):
    return {"available": INVENTORY[request["sku"]]["available"], "latency_ms": 40}


def quote_branch(request):
    quotes = [quote for quote in SUPPLIER_QUOTES[request["sku"]] if quote["trusted"] and quote["age_hours"] <= 48]
    best = min(quotes, key=lambda item: item["unit_price"])
    return {**best, "latency_ms": best["delay_ms"]}

with ThreadPoolExecutor(max_workers=2) as executor:
    inventory_future = executor.submit(inventory_branch, high_value)
    quote_future = executor.submit(quote_branch, high_value)
    inventory_result = inventory_future.result()
    quote_result = quote_future.result()

sequential_ms = inventory_result["latency_ms"] + quote_result["latency_ms"]
parallel_ms = max(inventory_result["latency_ms"], quote_result["latency_ms"])
print(f"Modeled sequential latency: {sequential_ms} ms")
print(f"Modeled parallel latency: {parallel_ms} ms")
assert parallel_ms < sequential_ms


## 3 - Map the Mechanism to LangGraph

Now the framework earns its place: nodes and conditional edges make the transition table executable and inspectable. The node bodies remain ordinary local functions.

```mermaid
flowchart TD
    E["resume router"] -->|"not gathered"| G["gather node"]
    E -->|"already gathered"| R["route node"]
    G --> R
    R -->|"auto"| A["auto approval"]
    R -->|"manager"| M["manager approval"]
    R -->|"finance"| F["finance approval"]
    A --> Z["END"]
    M --> Z
    F --> Z
    style E fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style R fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style A fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style M fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style Z fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```


In [ ]:
# -- Build the actual LangGraph --------------------------------------------
from langgraph.graph import END, StateGraph

class OrderState(TypedDict, total=False):
    request: dict
    inventory: dict
    quote: dict
    gathered: bool
    total: float
    route: str
    approved: bool
    terminal: str
    visits: list[str]


def graph_gather(state: OrderState):
    updated = gather_checks(state)
    return updated


def graph_route(state: OrderState):
    return route_approval(state)


def graph_approve(state: OrderState):
    return approve(state)

builder = StateGraph(OrderState)
builder.add_node("gather", graph_gather)
builder.add_node("route", graph_route)
builder.add_node("auto", graph_approve)
builder.add_node("manager", graph_approve)
builder.add_node("finance", graph_approve)
builder.set_entry_point("gather")
builder.add_edge("gather", "route")
builder.add_conditional_edges("route", lambda state: state["route"], {"auto": "auto", "manager": "manager", "finance": "finance"})
for node in ("auto", "manager", "finance"):
    builder.add_edge(node, END)
graph = builder.compile()

initial = {"request": high_value, "visits": [], "gathered": False, "approved": False}
graph_high = graph.invoke(initial)
print("LangGraph high-value visits:", graph_high["visits"])
assert graph_high["terminal"] == "completed" and "finance" in graph_high["visits"]


## 4 - Crash, Resume, and State Migration

A durable workflow resumes from persisted state, not from the beginning. The recovery router must recognize completed side effects. This local journal models the same invariant a production checkpointer enforces.

```mermaid
flowchart LR
    G["Gather complete"] --> J["Durable journal"]
    J --> X["Process crash"]
    X --> R["Reload state"]
    R --> P["Resume at approval"]
    P --> D["Done without re-gather"]
    style G fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style J fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style X fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style R fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style P fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```


In [ ]:
# -- Simulate crash recovery with a versioned journal ---------------------
gather_calls = 0

def counted_gather(state):
    global gather_calls
    gather_calls += 1
    return gather_checks(state)

pre_crash = {"request": high_value, "visits": [], "gathered": False, "approved": False, "state_version": 1}
pre_crash = counted_gather(pre_crash)
journal_payload = json.dumps(pre_crash)
del pre_crash

restored = json.loads(journal_payload)
assert restored["state_version"] == 1
if not restored["gathered"]:
    restored = counted_gather(restored)
restored = route_approval(restored)
restored = approve(restored)

print(f"Gather side-effect count across crash and resume: {gather_calls}")
print("Restored visits:", restored["visits"])
assert gather_calls == 1 and restored["terminal"] == "completed"
print("PASS: recovery resumed after quote collection without repeating the side effect.")


In [ ]:
# -- Final route health check across low and high values ------------------
def invoke_for(request):
    return graph.invoke({"request": request, "visits": [], "gathered": False, "approved": False})

low_run = invoke_for(low_value)
high_run = invoke_for(high_value)
assert "finance" not in low_run["visits"]
assert "finance" in high_run["visits"]
assert low_run["approved"] and high_run["approved"]
print("PASS: high-value work visits Finance; low-value work does not.")


## Roadmap Checkpoint

```mermaid
flowchart LR
    A["Durable workflow met"] --> B["Next: agentic retrieval"]
    style A fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

| Constraint | Before | After |
|---|---:|---:|
| Branch visibility | Hidden in loop | Explicit conditional edges |
| Parallel checks | Sequential modeled latency | Fan-out/fan-in uses slower branch bound |
| High-value approval | Not guaranteed | Finance node always visited |
| Crash after quote | Restart repeats work | Resume continues with one gather call |

### Coverage Ledger

| Tier | Covered here |
|---|---|
| Built and measured | State machine, fan-out/fan-in, LangGraph routing, crash journal, resume |
| Explained and illustrated | Reducers, interrupts, typed graph state |
| Named with a reason | Subgraphs and distributed checkpointers, deferred until one graph is understood |

### Key Takeaways

- Nodes own work; edges own control flow; state owns progress.
- Parallel reads need an explicit reducer before approval.
- Recovery starts from durable state, never from a hopeful retry.
- A framework is useful only after you can build the mechanism without it.
